<a href="https://colab.research.google.com/github/Elfikrie/Exploratory-Data-Analysis/blob/Po5/Praktikum5Saya_Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**3 Hal yang harus dihasilkan dari praktikum ini :**

1.   reports/data_quality_report.docx — laporan temuan dan keputusan
2.   otebooks/02_cleaning.ipynb — notebook pembersihan yang dapat dijalankan ulang
3. cleaning_log.md — catatan setiap keputusan beserta alasannya

In [246]:
import pandas as pd
import numpy as np
import hashlib, os

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 150)

FILE = 'bedtime_screentime_sleep_debt_kotor.csv'
if not os.path.exists(FILE):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError('Letakkan CSV di folder yang sama dengan notebook ini.')

mentah = pd.read_csv(FILE)
print('Data mentah:', mentah.shape)

Data mentah: (8600, 18)


In [247]:
#Mengamankan data mentah
d = mentah.copy()

jejak = []
def catat(langkah, terdampak, alasan):
    """Setiap perubahan dicatat SAAT dikerjakan, bukan belakangan."""
    jejak.append({'langkah': langkah, 'baris_terdampak': int(terdampak), 'alasan': alasan})
    print(f'  {langkah}: {int(terdampak)} baris')

print('Siap. Data mentah tetap utuh di variabel `mentah`.')


Siap. Data mentah tetap utuh di variabel `mentah`.


**Memastikan Tipe Data**

In [248]:
# Cek bentuk data
d.head(10)

,user_id,age,gender,occupation_type,chronotype,bedtime_phone_minutes,primary_bedtime_app,screen_brightness_pct,blue_light_filter_active,caffeine_post_5pm_mg,physical_activity_min,sleep_latency_min,total_sleep_hours,deep_sleep_pct,rem_sleep_pct,morning_alarm_snoozes,next_day_fatigue_score,sleep_debt_category
0,USR-00001,29,Female,Healthcare / Shift Worker,Intermediate,179,INSTAGRAM / REDDIT,54.0,1,0.0,21,84.4,3.20,23.5,22.2,7,10.0,Severe Sleep Debt
1,USR-00002,58,Female,Student,Intermediate,163,YouTube,76.0,1,92.0,72,86.7,4.06,18.0,21.5,7,10.0,Severe Sleep Debt
2,USR-00003,41,Female,Healthcare / Shift Worker,Night Owl,100,TikTok / Reels,59.0,0,45.0,23,70.5,3.20,18.1,20.5,7,10.0,Severe Sleep Debt
3,USR-00004,36,Non-Binary,Remote Tech,Intermediate,34,YouTube,52.0,0,97.0,49,38.1,6.99,21.6,22.4,1,1.7,Mild Deficit
4,USR-00005,23,Female,Healthcare / Shift Worker,Intermediate,27,TikTok / Reels,82.0,1,0.0,38,32.3,4.58,24.1,NaN,4,5.3,Moderate Debt
5,USR-00006,23,Male,Healthcare / Shift Worker,Intermediate,76,TikTok / Reels,39.0,No,136.0,42,50.6,5.87,21.4,17.5,4,5.0,Moderate Debt
6,USR-00007,20,Female,Remote Tech,Night Owl,53,TikTok / Reels,32.0,0,0.0,64,36.2,4.43,22.7,19.0,6,6.4,Moderate Debt
7,USR-00008,49,Male,Remote Tech,Morning Lark,113,YouTube,34.0,True,39.0,58,60.2,5.94,19.3,19.9,3,4.5,Moderate Debt
8,USR-00009,36,Male,Freelance / Creative,Morning Lark,65,News / Reading,35.0,1,0.0,25,28.2,8.00,22.2,18.5,1,1.0,Optimal Recovery
9,USR-00010,40,Female,Student,Night Owl,70,Streaming (Netflix/Hulu),63.0,1,115.0,59,53.7,4.70,23.5,16.6,4,7.3,Moderate Debt


In [249]:
# Pengecekan kembali
col = d['bedtime_phone_minutes'].astype(str)
print(col.str.replace(r'[-\d.]+', '', regex=True).str.strip().value_counts())

bedtime_phone_minutes
       8549
min      51
Name: count, dtype: int64


In [250]:
# Ubah data dan tipe
d['bedtime_phone_minutes'] = (
    d['bedtime_phone_minutes']
    .astype(str)
    .str.extract(r'(-?\d+\.?\d*)')[0]
    .astype(float)
)

print(d['bedtime_phone_minutes'].dtype)
print(d['bedtime_phone_minutes'].isna().sum(), 'NaN')
print(d['bedtime_phone_minutes'].describe())

float64
0 NaN
count    8600.000000
mean       59.204186
std        36.868346
min         1.000000
25%        31.000000
50%        51.000000
75%        79.000000
max       180.000000
Name: bedtime_phone_minutes, dtype: float64


In [251]:
# Cek type data : apakah sudah benar?
d.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8600 entries, 0 to 8599
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8600 non-null   object 
 1   age                       8600 non-null   object 
 2   gender                    8600 non-null   object 
 3   occupation_type           8429 non-null   object 
 4   chronotype                8473 non-null   object 
 5   bedtime_phone_minutes     8600 non-null   float64
 6   primary_bedtime_app       8600 non-null   object 
 7   screen_brightness_pct     8429 non-null   float64
 8   blue_light_filter_active  8600 non-null   object 
 9   caffeine_post_5pm_mg      8429 non-null   float64
 10  physical_activity_min     8600 non-null   int64  
 11  sleep_latency_min         8344 non-null   float64
 12  total_sleep_hours         8600 non-null   float64
 13  deep_sleep_pct            8343 non-null   float64
 14  rem_slee

## Teks, Placeholder, & **kategori**

In [252]:
# CONTOH 4.1 — buang spasi pada SELURUH kolom teks sekaligus
kolom_teks = [c for c in d.columns if d[c].dtype == object or str(d[c].dtype) == 'str']

n_spasi = 0
for kol in kolom_teks:
    s = d[kol].astype('string')
    n_spasi += int((s != s.str.strip()).sum())
    d[kol] = s.str.strip()

catat('Buang spasi awal dan akhir kolom teks', n_spasi,
      'Spasi tersembunyi memecah groupby dan melambungkan nunique')

  Buang spasi awal dan akhir kolom teks: 500 baris


In [253]:
# Penyeragaman kategori, dengan peta keputusan yang eksplisit
d['sleep_debt_category'] = d['sleep_debt_category'].astype('string').str.strip().str.title()

# Bagian yang HARUS diputuskan manusia, ditulis sebagai peta agar bisa ditinjau orang lain:
PETA_sleep_debt_category = {
    'Severe Sleep debt'        : 'Severe Sleep Debt',          # Pengetesan saja
}
sebelum = d['sleep_debt_category'].nunique()
d['sleep_debt_category'] = d['sleep_debt_category'].replace(PETA_sleep_debt_category)

catat('Seragamkan sleep_debt_category', sebelum - d['sleep_debt_category'].nunique(),
      'Huruf dan spasi oleh kode; sinonim dan salah ketik diputuskan manual lalu dicatat di peta')
print(sorted(d['sleep_debt_category'].dropna().unique()))

  Seragamkan sleep_debt_category: 0 baris
['Mild Deficit', 'Mild Defisit', 'Moderat Debt', 'Moderate Debt', 'Optimal Recovery', 'Optimal Recovry', 'Sever Sleep Debt', 'Severe Sleep Debt']


In [254]:
# Hasil Penyeragaman kategori
p5_kat      = d['sleep_debt_category']

print('P5_kat', p5_kat)


P5_kat 0       Severe Sleep Debt
1       Severe Sleep Debt
2       Severe Sleep Debt
3            Mild Deficit
4           Moderate Debt
              ...        
8595        Moderate Debt
8596    Severe Sleep Debt
8597         Mild Deficit
8598    Severe Sleep Debt
8599        Moderate Debt
Name: sleep_debt_category, Length: 8600, dtype: string


### DUPLIKAT

In [255]:
# duplikat persis
n_persis = int(d.duplicated().sum())
d = d.drop_duplicates()
catat('Hapus duplikat persis', n_persis, 'Seluruh kolom identik; satu baris disimpan')

  Hapus duplikat persis: 100 baris


In [256]:
# kunci utama yang kosong
n_kosong = int(d['user_id'].isna().sum())
d = d[d['user_id'].notna()]
catat('Buang baris tanpa user_id', n_kosong,
      'Kunci utama kosong tidak dapat ditelusuri kembali ke sumber')

  Buang baris tanpa user_id: 0 baris


## NILAI MUSTAHIL

In [257]:
# total_sleep_hours nol atau negatif: dijadikan kosong, bukan dibuang
n_j = int((d['total_sleep_hours'] <= 0).sum())
d.loc[d['total_sleep_hours'] <= 0, 'total_sleep_hours'] = np.nan
catat('total_sleep_hours <= 0 menjadi kosong', n_j,
      'Belum jelas retur atau salah input; barisnya masih berguna untuk kolom lain')

  total_sleep_hours <= 0 menjadi kosong: 5 baris


### **Nilai kosong, dan mengapa tidak ada pilihan yang netral**

In [258]:
# CONTOH 7.1 — eksperimen imputasi
if os.path.exists('P05_nilai_sebenarnya.csv'):
    truth = pd.read_csv('P05_nilai_sebenarnya.csv')
    asli = truth['total_sleep_hours']
    obs  = truth['total_sleep_hours']

    hasil = pd.DataFrame({
        'KEBENARAN (semua anggota)' : asli.describe()[['mean','50%','std','max']],
        'Hanya yang terlihat'       : obs.dropna().describe()[['mean','50%','std','max']],
        'Diisi dengan mean'         : obs.fillna(obs.mean()).describe()[['mean','50%','std','max']],
        'Diisi dengan median'       : obs.fillna(obs.median()).describe()[['mean','50%','std','max']],
    }).T.round(0)
    hasil.columns = ['mean', 'median', 'std', 'max']
    display(hasil.map(lambda v: f'{v:,.0f}'))
else:
    print('Berkas kebenaran tidak tersedia; lewati sel ini.')

Berkas kebenaran tidak tersedia; lewati sel ini.


### HASIL PEMBERSIHAN

In [259]:
d = d.reset_index(drop=True)

p5_baris_bersih  = len(d)
p5_dibuang       = len(mentah) - len(d)
p5_sleep_latency_min_kosong = int(d['sleep_latency_min'].isna().sum())

print('P5_baris_bersih', p5_baris_bersih)
print('P5_dibuang', p5_dibuang)
print('P5_sleep_latency_min_kosong', p5_sleep_latency_min_kosong)

P5_baris_bersih 8500
P5_dibuang 100
P5_sleep_latency_min_kosong 255


**MENYUSUN ULANG BENTUK TABEL**

In [260]:
# pivot: dari panjang ke lebar (untuk dibaca manusia)
lebar = (d.pivot_table(index='occupation_type', columns='sleep_debt_category', values='total_sleep_hours',
                       aggfunc='sum', observed=True)
          .fillna(0).round(0).astype(int))
lebar

sleep_debt_category,Mild Deficit,Mild Defisit,Moderat Debt,Moderate Debt,Optimal Recovery,Optimal Recovry,Sever Sleep Debt,Severe Sleep Debt
occupation_type,,,,,,,,
CORPORATE 9-TO-5,29,0,0,100,33,0,0,8
Corporate 9-to-5,4994,43,59,7581,4222,34,7,684
FREELANCE / CREATIVE,29,0,0,25,8,0,0,0
Freelance / Creative,1583,8,12,2451,1197,0,0,244
HEALTHCARE / SHIFT WORKER,8,0,0,48,9,0,0,7
Healthcare / Shift Worker,597,0,15,3476,122,0,0,601
REMOTE TECH,58,0,0,53,35,0,0,5
Remote Tech,3664,7,18,5882,3192,0,5,456
STUDENT,15,0,0,37,24,0,0,3


In [261]:
# merge: menggabungkan tabel lain
# Tabel ringkasan per anggota, lalu digabungkan kembali ke total tidur.
profil_anggota = (d.groupby('user_id')
                    .agg(jumlah_=('bedtime_phone_minutes', 'sum'),
                         total_tidur=('total_sleep_hours', 'sum'))
                    .reset_index())

gabung = d.merge(profil_anggota, on='user_id', how='left', validate='many_to_one')
print('Sebelum:', d.shape, '-> sesudah:', gabung.shape)
profil_anggota.head(3)

Sebelum: (8500, 18) -> sesudah: (8500, 20)


,user_id,jumlah_,total_tidur
0,USR-00001,179.0,3.20
1,USR-00002,163.0,4.06
2,USR-00003,100.0,3.20


## **Pipeline yang dapat dijalankan**

In [262]:
# CONTOH 10.1 — kerangka pipeline
def bersihkan(df_mentah):
    """Terima data mentah, kembalikan data bersih beserta jejak keputusannya."""
    x = df_mentah.copy()
    log = []
    def c(langkah, n, alasan):
        log.append({'langkah': langkah, 'baris_terdampak': int(n), 'alasan': alasan})

    # 1 tipe data
    # 2 spasi
    # 3 placeholder
    # 4 kategori
    # 5 duplikat
    # 6 nilai mustahil
    # 7 nilai kosong
    # 8 kolom turunan

    return x.reset_index(drop=True), pd.DataFrame(log)

print('Salin langkah-langkah di atas ke dalam fungsi ini untuk notebook M2 Anda.')

Salin langkah-langkah di atas ke dalam fungsi ini untuk notebook M2 Anda.


In [263]:
# CONTOH 10.2 — simpan hasil
os.makedirs('data/processed', exist_ok=True)
os.makedirs('reports', exist_ok=True)

d.to_csv('data/processed/bedtime_screen_sleep_debt_bersih.csv', index=False)
print('Tersimpan:', d.shape)
print()
print('Data mentah tidak disentuh:', mentah.shape)

Tersimpan: (8500, 18)

Data mentah tidak disentuh: (8600, 18)


### **Cleaning Log**

In [264]:
# cetak log
log = pd.DataFrame(jejak)
log.index = range(1, len(log) + 1)
log

,langkah,baris_terdampak,alasan
1,Buang spasi awal dan akhir kolom teks,500,Spasi tersembunyi memecah groupby dan melambun...
2,Seragamkan sleep_debt_category,0,Huruf dan spasi oleh kode; sinonim dan salah k...
3,Hapus duplikat persis,100,Seluruh kolom identik; satu baris disimpan
4,Buang baris tanpa user_id,0,Kunci utama kosong tidak dapat ditelusuri kemb...
5,total_sleep_hours <= 0 menjadi kosong,5,Belum jelas retur atau salah input; barisnya m...


In [265]:
# 1) Catat perubahan bedtime_phone_minutes ke log (hitung dari data mentah)
langkah_bp = "Konversi bedtime_phone_minutes ke numerik (buang satuan 'min')"

if langkah_bp not in log['langkah'].values:
    col = mentah['bedtime_phone_minutes']
    bp = pd.to_numeric(col, errors='coerce')
    n_terdampak = int((bp.isna() & col.notna()).sum())

    log.loc[log.index.max() + 1, ['langkah', 'baris_terdampak', 'alasan']] = [
        langkah_bp,
        n_terdampak,
        "Kolom bertipe teks karena sebagian nilai berformat '<angka> min'; "
        "diekstrak angkanya dengan regex dan diubah ke float agar bisa dihitung"
    ]

# 2) Simpan sebagai Markdown untuk repositori
baris = ['# Cleaning Log', '',
         'Dataset: bedtime_screen_sleep_debt.csv  ',
         f'Baris mentah: {len(mentah)} -> baris bersih: {len(d)}  ',
         '', '| # | Langkah | Baris terdampak | Alasan |', '|---|---|---|---|']
for i, r in log.iterrows():
    baris.append(f"| {i} | {r['langkah']} | {r['baris_terdampak']} | {r['alasan']} |")

with open('cleaning_log.md', 'w', encoding='utf-8') as f:
    f.write('\n'.join(baris))

print('\n'.join(baris[-5:]))   # tampilkan baris terakhir supaya langkah baru terlihat

| 2 | Seragamkan sleep_debt_category | 0.0 | Huruf dan spasi oleh kode; sinonim dan salah ketik diputuskan manual lalu dicatat di peta |
| 3 | Hapus duplikat persis | 100.0 | Seluruh kolom identik; satu baris disimpan |
| 4 | Buang baris tanpa user_id | 0.0 | Kunci utama kosong tidak dapat ditelusuri kembali ke sumber |
| 5 | total_sleep_hours <= 0 menjadi kosong | 5.0 | Belum jelas retur atau salah input; barisnya masih berguna untuk kolom lain |
| 6 | Konversi bedtime_phone_minutes ke numerik (buang satuan 'min') | 51.0 | Kolom bertipe teks karena sebagian nilai berformat '<angka> min'; diekstrak angkanya dengan regex dan diubah ke float agar bisa dihitung |


### **KELENGKAPAN LOG**

In [266]:
# jumlah langkah di cleaning log

p5_langkah   = len(jejak)
print('P5_langkah =', p5_langkah)

P5_langkah = 5


## **Verifikasi**

In [267]:
def diagnosa_singkat(data, nama):
    print(f'=== {nama} ===')
    print('  Baris                     :', len(data))
    print('  Duplikat persis           :', int(data.duplicated().sum()))
    print('  Kategori unik             :', data['occupation_type'].astype(str).nunique())
    print('  id_anggota unik           :', data['user_id'].astype(str).nunique())
    print('  total_sleep_hours numerik :', pd.api.types.is_numeric_dtype(data['total_sleep_hours']))

    # Konversi aman: teks non-angka -> NaN
    bp = pd.to_numeric(data['bedtime_phone_minutes'], errors='coerce')
    print('  bedtime_phone_minutes tipe asli         :', data['bedtime_phone_minutes'].dtype)
    print('  bedtime_phone_minutes non-angka (jadi NaN):', int(bp.isna().sum() - data['bedtime_phone_minutes'].isna().sum()))
    print('  bedtime_phone_minutes <= 0              :', int((bp <= 0).sum()))
    print()

diagnosa_singkat(mentah, 'SEBELUM')
diagnosa_singkat(d, 'SESUDAH')

=== SEBELUM ===
  Baris                     : 8600
  Duplikat persis           : 100
  Kategori unik             : 26
  id_anggota unik           : 8500
  total_sleep_hours numerik : True
  bedtime_phone_minutes tipe asli         : object
  bedtime_phone_minutes non-angka (jadi NaN): 51
  bedtime_phone_minutes <= 0              : 0

=== SESUDAH ===
  Baris                     : 8500
  Duplikat persis           : 0
  Kategori unik             : 16
  id_anggota unik           : 8500
  total_sleep_hours numerik : True
  bedtime_phone_minutes tipe asli         : float64
  bedtime_phone_minutes non-angka (jadi NaN): 0
  bedtime_phone_minutes <= 0              : 0



In [268]:
d.head(5)

,user_id,age,gender,occupation_type,chronotype,bedtime_phone_minutes,primary_bedtime_app,screen_brightness_pct,blue_light_filter_active,caffeine_post_5pm_mg,physical_activity_min,sleep_latency_min,total_sleep_hours,deep_sleep_pct,rem_sleep_pct,morning_alarm_snoozes,next_day_fatigue_score,sleep_debt_category
0,USR-00001,29,Female,Healthcare / Shift Worker,Intermediate,179.0,INSTAGRAM / REDDIT,54.0,1,0.0,21,84.4,3.20,23.5,22.2,7,10.0,Severe Sleep Debt
1,USR-00002,58,Female,Student,Intermediate,163.0,YouTube,76.0,1,92.0,72,86.7,4.06,18.0,21.5,7,10.0,Severe Sleep Debt
2,USR-00003,41,Female,Healthcare / Shift Worker,Night Owl,100.0,TikTok / Reels,59.0,0,45.0,23,70.5,3.20,18.1,20.5,7,10.0,Severe Sleep Debt
3,USR-00004,36,Non-Binary,Remote Tech,Intermediate,34.0,YouTube,52.0,0,97.0,49,38.1,6.99,21.6,22.4,1,1.7,Mild Deficit
4,USR-00005,23,Female,Healthcare / Shift Worker,Intermediate,27.0,TikTok / Reels,82.0,1,0.0,38,32.3,4.58,24.1,<NA>,4,5.3,Moderate Debt


In [271]:
d.info()
d.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8500 entries, 0 to 8499
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8500 non-null   string 
 1   age                       8500 non-null   string 
 2   gender                    8500 non-null   string 
 3   occupation_type           8330 non-null   string 
 4   chronotype                8373 non-null   string 
 5   bedtime_phone_minutes     8500 non-null   float64
 6   primary_bedtime_app       8500 non-null   string 
 7   screen_brightness_pct     8330 non-null   float64
 8   blue_light_filter_active  8500 non-null   string 
 9   caffeine_post_5pm_mg      8330 non-null   float64
 10  physical_activity_min     8500 non-null   int64  
 11  sleep_latency_min         8245 non-null   float64
 12  total_sleep_hours         8495 non-null   float64
 13  deep_sleep_pct            8245 non-null   float64
 14  rem_slee

(8500, 18)